# Simple script to run benchmarks

The script draws time graphs for the benchmark data for our driver, the datastax driver and the rust-driver.

Parameters:
- `n_min, n_max, step` - range of the number of queries to the database with a step added at each iteration.
- `repeat` - Number of test calls for n.

Result of the script is a `graph.svg` saved in the `benchmarks/` directory

In [1]:
from subprocess import run
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --------- parameters ------------
repeat = 5
n_min = 10
n_max = 1_000
step = 1000

# --------- libs and rust benchmark names ----------
libs = ["scylladb-javascript-driver", "cassandra-driver"]
#benchmarks = ["concurent_insert.js", "insert.js", "select.js"]
benchmarks = ["concurent_insert.js"]
name_rust = {}
name_rust["concurent_insert.js"] = "concurrent_insert_benchmark"
name_rust["insert.js"] = "insert_benchmark"
name_rust["select.js"] = "select_benchmark"

In [2]:
# Function to parse the output from the time function
def parse_time(s):
    q = s.split('\t')[1][:-1]
    q2 = q.split('m')
    return int(q2[0]), float(q2[1].replace(',', '.'))

In [3]:
df = {}
for ben in benchmarks:
    df[ben] = pd.DataFrame(columns=['n', libs[0], libs[1], 'rust-driver'])
    
    # Build rust benchmark
    data = run("cargo build --bin "+name_rust[ben]+" -r",
                           capture_output=True, shell=True, text=True, executable='/bin/bash')
    if data.returncode != 0:
        raise Exception("Build error: " + name_rust[ben]) 
    
    for n in range(n_min, n_max+1 + n_min, step):
        dict_for_n = {}
        dict_for_n['n'] = n
        results = []
        # ------ rust -------
        for _ in range(repeat):
            data = run("time CNT="+str(n)+" cargo run --bin "+name_rust["insert.js"]+" -r ",
                               capture_output=True, shell=True, text=True, executable='/bin/bash')
            
            m, s = parse_time(data.stderr.split('\n')[-4])
            offset = float(data.stderr.split('\n')[0].split('in ')[-1][:-1]) # ugly offset calculation
            
            results.append(float(m * 60) + s - offset)
        dict_for_n["rust-driver"] = results   
        
        # ------ node -----
        for lib in libs:
            results = []
            for _ in range(repeat):
                data = run("time node logic/"+ben+" "+str(lib)+" "+str(n),
                                   capture_output=True, shell=True, text=True, executable='/bin/bash')
                m, s = parse_time(data.stderr.split('\n')[-4])
                results.append(float(m * 60) + s)
            dict_for_n[lib] = results
        # print(ben, dict_for_n)
        df[ben].loc[len(df[ben])] = dict_for_n

Exception: Build error: concurrent_insert_benchmark

# Drawing a graph



In [ ]:
rows = (len(df) + 2) // 3

fig, axes = plt.subplots(rows, 3, figsize=(15, 5 * rows), facecolor="white")

fig.suptitle("Time", fontsize=16, fontweight="bold", y=0.99)

axes = axes.flatten()
libs.append("rust-driver")
for ax, (test_name, data) in zip(axes, df.items()):
    ax.set_facecolor("white")

    for lib in libs:
        data[f"{lib}_mean"] = data[lib].apply(np.mean)
        data[f"{lib}_std"] = data[lib].apply(np.std)

        ax.errorbar(data["n"], data[f"{lib}_mean"], yerr=data[f"{lib}_std"], label=lib, linestyle="-", linewidth=2, capsize=5)

    ax.set_xlabel("Number of requests")
    ax.set_ylabel("Time [s]")
    ax.set_title(f"Benchmark - {test_name.split('.')[0]}")
    ax.legend()

plt.style.use('default')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("graph.svg", format="svg", dpi=300)
plt.show()